# 01 · Configuración del dataset

**Objetivos de aprendizaje**

- Seleccionar el archivo de datos que se analizará durante el taller.
- Comprender las opciones de lectura de Spark: formato, separador, encabezado, codificación y tipos.
- Validar la carga y diagnosticar problemas frecuentes.
- Guardar la configuración que utilizarán los demás cuadernos.

| Aspecto | Detalle |
|---|---|
| Duración estimada | 25 minutos |
| Requisito previo | Cuaderno 00 |
| Entrada | Archivo de datos (por defecto, el dataset de ejemplo) |
| Salida | `data/dataset_config.json` |

### Cómo utilizar este cuaderno

1. Las celdas se ejecutan **en orden**, de arriba hacia abajo, con `Shift + Enter`. Cada celda puede depender de variables creadas en celdas anteriores.
2. Las celdas que comienzan con `# >>> CELDA EDITABLE` son las únicas que deben modificarse para adaptar el análisis a otro dataset. El resto del código se ajusta automáticamente a las columnas detectadas.
3. Cada sección sigue la misma estructura: **fundamento** (qué es y por qué se hace), **código**, **interpretación** del resultado y **adaptación** al dataset propio.
4. Ante un error `NameError` (variable no definida), lo habitual es que no se haya ejecutado una celda anterior. La solución es *Kernel > Restart Kernel and Run All Cells*.
5. Mientras la sesión de Spark está activa, la Spark UI (normalmente <http://localhost:4040>) muestra los *jobs*, *stages* y tareas ejecutados.

La guía completa de uso de los cuadernos se encuentra en `docs/08-uso-de-notebooks.md`.

## 1. El dataset de ejemplo

El taller incluye un dataset inicial, `data/ejemplo/ventas_tienda.csv`, que sirve de guía para todos los cuadernos. Contiene 20 000 ventas sintéticas de una tienda en ciudades de Colombia durante 2025, con 15 columnas:

| Columna | Tipo | Descripción |
|---|---|---|
| `id_venta` | entero | Identificador de la venta |
| `fecha` | fecha | Fecha de la venta (aaaa-mm-dd) |
| `ciudad`, `region` | texto | Ubicación |
| `canal`, `metodo_pago` | texto | Canal de venta y medio de pago |
| `categoria`, `producto` | texto | Producto vendido |
| `cantidad`, `precio_unitario`, `descuento` | numéricos | Detalle de la venta |
| `edad_cliente`, `genero_cliente`, `segmento_cliente` | mixtos | Datos del cliente |
| `total` | numérico | Valor total de la venta (variable objetivo sugerida) |

Para practicar la limpieza de datos, el archivo contiene de forma intencional alrededor de un 2 % de registros con problemas: valores faltantes, cantidades negativas, filas duplicadas y nombres de ciudad con espacios o mayúsculas inconsistentes. Así ocurre con frecuencia en los datos reales: Wickham (2014) señala que una parte considerable del tiempo de análisis se dedica a preparar los datos.

**Recomendación:** recorrer todo el taller primero con el dataset de ejemplo y, después, repetirlo con un dataset propio.

## Preparación del entorno

La celda siguiente realiza cuatro tareas:

1. **Ubica la carpeta del taller** con `ubicar_taller()`, que busca `scripts/taller_spark.py` a partir de la carpeta actual. Esto evita errores de ruta del tipo `Input path does not exist: file:/data/...`, que aparecen cuando el cuaderno se abre desde una ubicación distinta, como ocurre en Google Colab, donde la carpeta de trabajo es `/content`. Después fija la carpeta `notebooks/` como carpeta de trabajo.
2. **Solo en Google Colab**: busca el taller en cualquier carpeta de `/content` y en Google Drive; si no lo encuentra, lo descarga con `git clone` desde la dirección indicada en `REPO_URL` (por eso, en Colab, debe reemplazarse `<usuario>` por el usuario de GitHub donde está publicado el taller). También instala las dependencias que falten en Colab.
3. Agrega la carpeta `scripts/` a la ruta de búsqueda de Python, para poder importar el módulo `taller_spark.py`, que contiene las funciones de apoyo del taller.
4. Importa `functions` de PySpark con el alias `F`, convención habitual en la documentación oficial (The Apache Software Foundation [ASF], 2026b).
Finalmente, crea la `SparkSession` mediante `iniciar_spark()`. Esta función verifica que Java sea la versión 17 o 21, utiliza el mismo intérprete de Python en el *driver* y en los *executors* y se conecta al clúster de Docker cuando existe la variable `SPARK_MASTER`; en cualquier otro caso trabaja en modo `local[*]`.

In [ ]:
# Preparación del entorno: se ejecuta siempre en primer lugar
import glob, os, subprocess, sys
from pathlib import Path

# >>> CELDA EDITABLE (solo Google Colab): dirección del repositorio del taller en GitHub
REPO_URL = "https://github.com/<usuario>/taller-apache-spark.git"

def en_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

def ubicar_taller():
    """Devuelve la carpeta raíz del taller: la que contiene scripts/taller_spark.py."""
    actual = Path.cwd().resolve()
    candidatos = [actual, *actual.parents]
    for patron in ("/content/*/scripts/taller_spark.py",                  # clonado en Colab
                   "/content/drive/MyDrive/*/scripts/taller_spark.py",    # copiado en Google Drive
                   "/content/drive/MyDrive/*/*/scripts/taller_spark.py"):
        candidatos += [Path(p).parent.parent for p in sorted(glob.glob(patron))]
    for base in candidatos:
        if (base / "scripts" / "taller_spark.py").exists():
            return base
    if en_colab():
        if "<usuario>" in REPO_URL:
            raise RuntimeError("En Google Colab debe escribirse en REPO_URL la dirección del repositorio "
                               "del taller (reemplazar <usuario>) y volver a ejecutar esta celda.")
        destino = Path("/content") / Path(REPO_URL.rstrip("/")).stem        # nombre del repositorio
        subprocess.run(["git", "clone", "-q", REPO_URL, str(destino)], check=True)
        return destino
    raise FileNotFoundError("No se encontró la carpeta del taller (scripts/taller_spark.py). "
                            "Abrir el cuaderno desde la carpeta notebooks/ del repositorio.")

RAIZ_TALLER = ubicar_taller()
os.chdir(RAIZ_TALLER / "notebooks")              # rutas iguales en cualquier entorno (local, Docker, Colab)
sys.path.insert(0, str(RAIZ_TALLER / "scripts"))

if en_colab():                                    # en Colab se instalan las dependencias que falten
    import importlib.util
    faltan = [m for m in ("openpyxl",) if importlib.util.find_spec(m) is None]
    try:
        import pyspark
        if int(pyspark.__version__.split(".")[0]) < 4:
            faltan.append("pyspark==4.3.0")
    except ImportError:
        faltan.append("pyspark==4.3.0")
    if faltan:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan], check=True)
print("Carpeta del taller:", RAIZ_TALLER)

from taller_spark import *                       # iniciar_spark, cargar_desde_config, perfil, ...
from pyspark.sql import functions as F           # funciones de columnas de Spark

spark = iniciar_spark("01_Configuracion")

## 2. Archivos disponibles

La función `listar_datasets()` busca archivos con extensiones admitidas dentro de `data/`. Si se cargó un archivo propio en `data/mis_datos/` y no aparece, debe revisarse la extensión del archivo.

In [ ]:
for ruta in listar_datasets():
    print(ruta)

## 3. Definición de la configuración

La configuración es un diccionario de Python. Cada clave controla un aspecto de la lectura o del análisis:

| Clave | Valores | Significado |
|---|---|---|
| `nombre` | texto sin espacios | Nombre corto del dataset; se usa para nombrar los archivos de salida |
| `descripcion` | texto | Descripción breve (se recomienda incluir la fuente) |
| `ruta` | texto | Ruta del archivo desde la raíz del repositorio, por ejemplo `data/mis_datos/archivo.csv` |
| `formato` | `auto`, `csv`, `json`, `parquet`, `orc`, `excel` | Con `auto` se deduce de la extensión |
| `separador` | `auto`, `","`, `";"`, `"\t"`, `"\|"` | Delimitador de columnas en archivos de texto |
| `encabezado` | `True` / `False` | Si la primera fila contiene los nombres de las columnas |
| `codificacion` | `auto`, `UTF-8`, `ISO-8859-1` | Juego de caracteres; los archivos de Excel en Windows suelen ser `ISO-8859-1` |
| `decimal` | `auto`, `"."`, `","` | Separador decimal; con `,` se convierten valores como `1.234,56` |
| `multilinea` | `True` / `False` | Activar si hay textos con saltos de línea dentro de comillas |
| `hoja_excel` | número o nombre | Hoja que se lee en archivos de Excel |
| `columna_fecha` | nombre o `None` | Fecha principal para análisis temporales |
| `formato_fecha` | `None` o patrón | Con `None` se detecta; ejemplos: `dd/MM/yyyy`, `yyyy-MM-dd HH:mm:ss` |
| `columna_objetivo` | nombre o `None` | Variable que se intentará predecir en el cuaderno 06 |
| `columnas_excluir` | lista | Identificadores u otras columnas que no deben analizarse |
| `max_categorias` | número | Máximo de valores distintos para tratar una columna de texto como categórica |
| `nombre_tabla` | texto | Nombre de la vista para las consultas SQL del cuaderno 05 |

Los nombres de columna pueden escribirse tal como aparecen en el archivo: el taller los normaliza (minúsculas, sin tildes ni espacios), de modo que `Fecha de Compra` se convierte en `fecha_de_compra`.

**Adaptación al dataset propio.** En la celda siguiente se modifican, como mínimo, `nombre`, `ruta`, `columna_fecha`, `columna_objetivo` y `columnas_excluir`. Las demás opciones pueden quedar en `auto`.

In [ ]:
# >>> CELDA EDITABLE: configuración del dataset
CONFIG = {
    "nombre": "ventas_tienda",
    "descripcion": "Ventas de una tienda en ciudades de Colombia, 2025 (dataset sintético de ejemplo).",
    "ruta": "data/ejemplo/ventas_tienda.csv",
    "formato": "auto",
    "separador": "auto",
    "encabezado": True,
    "codificacion": "auto",
    "decimal": "auto",
    "multilinea": False,
    "hoja_excel": 0,
    "columna_fecha": "fecha",
    "formato_fecha": None,
    "columna_objetivo": "total",
    "columnas_excluir": ["id_venta"],
    "max_categorias": 50,
    "nombre_tabla": "datos",
}

Como referencia, la configuración de un archivo exportado desde Excel en español, con punto y coma como separador y coma decimal, podría ser:

```python
CONFIG = {
    "nombre": "matriculas_2025",
    "descripcion": "Matrículas por programa (fuente: portal institucional, 2025).",
    "ruta": "data/mis_datos/matriculas.csv",
    "separador": ";",
    "codificacion": "ISO-8859-1",
    "decimal": ",",
    "columna_fecha": "Fecha de Matrícula",
    "formato_fecha": "dd/MM/yyyy",
    "columna_objetivo": "Valor Pagado",
    "columnas_excluir": ["Documento"],
    ...
}
```

## 4. Carga de prueba

`cargar_dataset()` aplica la configuración y ejecuta, internamente, la lectura de Spark correspondiente. Para un CSV, el equivalente directo en PySpark sería:

```python
spark.read.option("header", True).option("inferSchema", True) \
     .option("sep", ",").option("encoding", "UTF-8").csv("ruta/archivo.csv")
```

La opción `inferSchema` hace que Spark recorra los datos para deducir el tipo de cada columna (ASF, 2026b). Además de leer, la función detecta el separador y la codificación cuando están en `auto`, normaliza los nombres de columna, convierte la coma decimal y transforma en fechas las columnas de texto que lo parecen. El diccionario `info` resume lo que se detectó.

In [ ]:
df, info = cargar_dataset(spark, **parametros_carga(CONFIG))
info

### Esquema y primeras filas

`printSchema()` muestra el nombre, el tipo y la posibilidad de valores nulos de cada columna. `show(5)` presenta las primeras cinco filas. Ambas operaciones se ejecutan sin traer todo el dataset a la memoria del equipo.

In [ ]:
df.printSchema()

In [ ]:
df.show(5, truncate=30)

In [ ]:
print(f"Filas: {df.count():,} | Columnas: {len(df.columns)}")

**Interpretación.** Deben comprobarse tres aspectos:

1. **Número de columnas**: si aparece una sola columna con todo el texto, el separador es incorrecto.
2. **Tipos**: las cantidades y los valores deben ser `int` o `double`; las fechas, `date` o `timestamp`. Un número leído como `string` suele indicar coma decimal o valores no numéricos (por ejemplo, `N/A`).
3. **Caracteres**: las tildes y la `ñ` deben verse correctamente; si aparecen símbolos como `Ã©`, la codificación no es la adecuada.

## 5. Diagnóstico automático

`diagnosticar_carga()` revisa estos problemas y también comprueba que las columnas indicadas en la configuración existan. Si la lista está vacía, la carga es correcta. En caso contrario, se ajusta la celda de configuración y se vuelven a ejecutar las secciones 4 y 5.

In [ ]:
avisos = diagnosticar_carga(df, CONFIG)
if avisos:
    for a in avisos:
        print("[AVISO]", a)
else:
    print("[OK] No se detectaron problemas de carga.")

## 6. Roles de las columnas

Los cuadernos siguientes deciden qué análisis aplicar según el rol de cada columna. La tabla de perfil resume, por columna, el tipo de Spark, el rol asignado, la cantidad y el porcentaje de nulos, el número aproximado de valores distintos y los valores mínimo y máximo.

El conteo de distintos utiliza `approx_count_distinct`, un algoritmo aproximado (HyperLogLog++) que evita ordenar todos los datos y es mucho más rápido en grandes volúmenes (ASF, 2026b).

In [ ]:
roles = clasificar_columnas(df, excluir=CONFIG["columnas_excluir"], max_categorias=CONFIG["max_categorias"])
resumen_roles(roles)
perfil(df, roles)

**Interpretación con el dataset de ejemplo.** La columna `ciudad` presenta más valores distintos de los esperados (12 ciudades) debido a las variantes con espacios y mayúsculas; este problema se corrige en el cuaderno 04. Las columnas `cantidad` y `total` tienen nulos y valores negativos, que también se tratarán en la limpieza.

**Adaptación.** Si una columna con códigos numéricos (por ejemplo, un código de municipio) aparece como numérica, conviene agregarla a `columnas_excluir` o convertirla en texto. Si una columna de texto con muchas categorías debería tratarse como categórica, se aumenta `max_categorias`.

## 7. Guardar la configuración

Al guardar, la configuración se escribe en `data/dataset_config.json`. Los cuadernos 02 a 07 la leen con `cargar_desde_config()`, por lo que no es necesario repetir las opciones de lectura en cada uno.

Al cambiar de dataset se elimina la referencia a la versión limpia anterior, de modo que debe ejecutarse de nuevo el cuaderno 04.

In [ ]:
guardar_config(CONFIG)
leer_config()

Para volver en cualquier momento al dataset de ejemplo, se elimina el símbolo `#` de la línea siguiente y se ejecuta la celda.

In [ ]:
# guardar_config(CONFIG_EJEMPLO)

## Ejercicios propuestos

1. Cambiar `separador` por `";"` en la configuración, ejecutar la carga y describir qué ocurre con el número de columnas. Restaurar luego el valor `auto`.
2. Cargar un dataset propio, revisar el diagnóstico y documentar en una celda Markdown qué opciones fue necesario ajustar y por qué.
3. Registrar en la `descripcion` la fuente del dataset propio con formato APA. Para conjuntos de datos, la forma general es: Autor. (Año). *Título del conjunto de datos* (Versión) [Conjunto de datos]. Editorial. URL (American Psychological Association, 2020).

In [ ]:
spark.stop()

## Referencias

American Psychological Association. (2020). *Publication manual of the American Psychological Association* (7.ª ed.). https://doi.org/10.1037/0000165-000

The Apache Software Foundation. (2026b). *Spark SQL, DataFrames and Datasets guide* (Versión 4.3.0) [Documentación técnica]. https://spark.apache.org/docs/latest/sql-programming-guide.html

Wickham, H. (2014). Tidy data. *Journal of Statistical Software, 59*(10), 1–23. https://doi.org/10.18637/jss.v059.i10